In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

↑今どんなファイルがあるか読み込み

In [ ]:
train_df = pd.read_csv("../input/house-prices-advanced-regression-techniques/train.csv")
test_df = pd.read_csv("../input/house-prices-advanced-regression-techniques/test.csv")
sample_df = pd.read_csv("../input/house-prices-advanced-regression-techniques/sample_submission.csv")
#各ファイルを読み込み

In [ ]:
train_df

In [ ]:
test_df

In [ ]:
sample_df

In [ ]:
X = train_df.drop(["Id", "SalePrice"], axis=1)
#[["MSSubClass","MSZoning","LotFrontage","LotArea","Street","Alley","LotShape","LandContour","Utilities","LotConfig","LandSlope","Neighborhood","Condition1","Condition2","BldgType","HouseStyle","OverallQual","OverallCond","YearBuilt","YearRemodAdd","RoofStyle","RoofMatl","Exterior1st","Exterior2nd","MasVnrType","MasVnrArea","ExterQual","ExterCond","Foundation","BsmtQual","BsmtCond","BsmtExposure","BsmtFinType1","BsmtFinSF1","BsmtFinType2","BsmtFinSF2","BsmtUnfSF","TotalBsmtSF","Heating","HeatingQC","CentralAir","Electrical","1stFlrSF","2ndFlrSF","LowQualFinSF","GrLivArea","BsmtFullBath","BsmtHalfBath","FullBath","HalfBath","BedroomAbvGr","KitchenAbvGr","KitchenQual","TotRmsAbvGrd","Functional","Fireplaces","FireplaceQu","GarageType","GarageYrBlt","GarageFinish","GarageCars","GarageArea","GarageQual","GarageCond","PavedDrive","WoodDeckSF","OpenPorchSF","EnclosedPorch","3SsnPorch","ScreenPorch","PoolArea","PoolQC","Fence","MiscFeature","MiscVal","MoSold","YrSold","SaleType","SaleCondition"
#]]
y = np.log1p(train_df["SalePrice"])
X_test_1 = test_df.drop("Id", axis=1)
#[["MSSubClass","MSZoning","LotFrontage","LotArea","Street","Alley","LotShape","LandContour","Utilities","LotConfig","LandSlope","Neighborhood","Condition1","Condition2","BldgType","HouseStyle","OverallQual","OverallCond","YearBuilt","YearRemodAdd","RoofStyle","RoofMatl","Exterior1st","Exterior2nd","MasVnrType","MasVnrArea","ExterQual","ExterCond","Foundation","BsmtQual","BsmtCond","BsmtExposure","BsmtFinType1","BsmtFinSF1","BsmtFinType2","BsmtFinSF2","BsmtUnfSF","TotalBsmtSF","Heating","HeatingQC","CentralAir","Electrical","1stFlrSF","2ndFlrSF","LowQualFinSF","GrLivArea","BsmtFullBath","BsmtHalfBath","FullBath","HalfBath","BedroomAbvGr","KitchenAbvGr","KitchenQual","TotRmsAbvGrd","Functional","Fireplaces","FireplaceQu","GarageType","GarageYrBlt","GarageFinish","GarageCars","GarageArea","GarageQual","GarageCond","PavedDrive","WoodDeckSF","OpenPorchSF","EnclosedPorch","3SsnPorch","ScreenPorch","PoolArea","PoolQC","Fence","MiscFeature","MiscVal","MoSold","YrSold","SaleType","SaleCondition"
#]]

In [ ]:
X

In [ ]:
y

In [ ]:
X_test_1

In [ ]:
#X = train_df[["LotFrontage","OverallQual","LotArea","MSSubClass","OverallCond","YearBuilt","YearRemodAdd","1stFlrSF","2ndFlrSF","LowQualFinSF","GrLivArea","FullBath","HalfBath","BedroomAbvGr","KitchenAbvGr","TotRmsAbvGrd","Fireplaces","WoodDeckSF","OpenPorchSF","EnclosedPorch","3SsnPorch","ScreenPorch","PoolArea","MiscVal","MoSold","YrSold"]]
#y = np.log1p(train_df["SalePrice"])
#X_test = test_df[["LotFrontage","OverallQual","LotArea","MSSubClass","OverallCond","YearBuilt","YearRemodAdd","1stFlrSF","2ndFlrSF","LowQualFinSF","GrLivArea","FullBath","HalfBath","BedroomAbvGr","KitchenAbvGr","TotRmsAbvGrd","Fireplaces","WoodDeckSF","OpenPorchSF","EnclosedPorch","3SsnPorch","ScreenPorch","PoolArea","MiscVal","MoSold","YrSold"]]

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
# 外れ値の除去 (GrLivArea > 4000 かつ ｙ(log(SalePrice)) < 12.5程度の物件2件)
outliers_to_remove = X[(X['GrLivArea'] > 4000) & (y < 12.5)].index
# 訓練データと目的変数から該当する行を削除
X = X.drop(outliers_to_remove)
y = y.drop(outliers_to_remove)

In [ ]:
ntrain_size = len(X)
all_data = pd.concat([X, X_test_1], axis=0, ignore_index=True)
print(f"\n結合後: {all_data.shape}")
#欠損値0埋め
all_data = all_data.fillna(0)
all_data = pd.get_dummies(all_data, drop_first=True)
print(f"ダミー変数化後: {all_data.shape}")

In [ ]:
# 4. trainとtestに分割
X_processed = all_data[:ntrain_size]      # 最初の1460行
X_test_processed = all_data[ntrain_size:] # 残りの1459行

In [ ]:
X_processed= X_processed.fillna(0)
X_test_processed = X_test_processed.fillna(0)

In [ ]:
X_processed

In [ ]:
print(f"\nX_processed: {X_processed.shape}")      # (1460, 約300)
print(f"X_test_processed: {X_test_processed.shape}") # (1459, 約300)

In [ ]:
# ===== 5. train_test_splitは1回だけ実行 =====
#X_train, X_valid, y_train, y_valid = train_test_split(X_processed, y, random_state=0)
X_train, X_valid, y_train, y_valid = train_test_split(X_processed, y, test_size=0.2, random_state=42)

In [ ]:
print(f"\nX_train: {X_train.shape}")   # (1095, 約300)
print(f"X_valid: {X_valid.shape}")     # (365, 約300)
print(f"y_train: {y_train.shape}")     # (1095,)
print(f"y_valid: {y_valid.shape}")     # (365,)

In [ ]:
#X_train, X_valid, y_train, y_valid = train_test_split(X, y, random_state=0)
#モデルを作る

### モデル構築

In [ ]:
#from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso

In [ ]:
#model = LinearRegression()

In [ ]:
#model.fit(X_train, y_train)

One hot Encoding

In [ ]:
#テキスト→数値化
#X_2 = pd.get_dummies(X)
#X_test_2 = pd.get_dummies(X_test)

In [ ]:
#X_2

In [ ]:
#X_test_2

In [ ]:
#０埋め
#X_3= X_2.fillna(0)
#X_test_3= X_test_2.fillna(0)

In [ ]:
#model = LinearRegression()
model = Lasso(alpha = 0.001)
model.fit(X_train, y_train)

### モデル適用と評価

In [ ]:
pred_train = model.predict(X_train)
pred_valid = model.predict(X_valid)
pred_test = model.predict(X_test_processed)
#train:学習データ、valid:検証データ、test:テスト用データをモデルと比較

In [ ]:
from sklearn.metrics import mean_squared_error

In [ ]:
print(mean_squared_error(y_train, pred_train), mean_squared_error(y_valid, pred_valid))
#予測値の作成

### Submission

In [ ]:
sub_df = pd.DataFrame({
    "Id": sample_df["Id"], 
    "SalePrice": np.expm1(pred_test)
})
#投稿ファイルに合うように変形

In [ ]:
sub_df.to_csv("simple.csv", index=False)

In [ ]:
sub_df